# Phase 4 - Evaluate the U-Net's robustness to GAN-generated fog

We have:
- A U-Net trained on **VDD clean** (Phase 1's v2 model with class weights, val mIoU 0.7602, test mIoU 0.7168)
- Four foggy variants of VDD generated by Pix2Pix (Phase 3):
  - `VDD_foggy_medium_256`: medium fog applied at 256x256, saved at 768x768
  - `VDD_foggy_medium_768`: medium fog applied at 768x768, saved at 768x768
  - `VDD_foggy_dense_256` : dense fog applied at 256x256, saved at 768x768
  - `VDD_foggy_dense_768` : dense fog applied at 768x768, saved at 768x768

**Goal**: measure how much the U-Net's mIoU drops on each foggy variant
compared to the clean baseline. This is the central result of the project.

Expected behavior:
- mIoU drops on all foggy variants (the U-Net never saw fog during training)
- dense fog hurts more than medium
- per-class IoU drops asymmetrically: small / textured classes (vehicle) likely
  hurt more than large uniform classes (water, vegetation)

**Tools**: we already have `src/evaluation/evaluate.py` (writes JSON + TB).
We just call it 5 times with different `--data-root`.

**Tip**: this notebook is fast (~5 min). If your Phase 3 Colab session is still
alive and has VDD + the 4 foggy datasets in `/content/data/`, you can skip
the copy steps (4) and jump straight to (8). Otherwise run all sections.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the project from GitHub

In [ ]:
import os

if not os.path.isdir('/content/fog-uav-robustness'):
    !git clone https://github.com/FabrizioCozzolino/fog-uav-robustness.git /content/fog-uav-robustness
%cd /content/fog-uav-robustness
!ls

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/fog-uav-robustness'

## 4. Restore VDD + regenerate the 4 foggy variants

We regenerate the foggy datasets here because the previous version had masks
at the original VDD resolution (4000x3000), which conflicts with the foggy
images saved at 768x768 -- Albumentations requires equal H/W between image
and mask.

This cell does the following in sequence:
  1. Removes any corrupted local VDD (had only a few images)
  2. Copies VDD fresh from Drive
  3. Removes the old foggy datasets (with mask shape mismatch)
  4. Restores the two Pix2Pix generators from Drive
  5. Regenerates the 4 foggy variants with the fixed script (mask at 768x768)
  6. Backs up the new foggy datasets to Drive (overwriting the old ones)

Total time: ~25-30 minutes (most of it is the VDD copy).

In [ ]:
import os, time, shutil

# ===== Step A: clean up local VDD if corrupt, recopy from Drive =====
LOCAL_VDD = '/content/data/VDD'
DRIVE_VDD = '/content/drive/MyDrive/fog-uav-robustness/data/VDD'

needs_recopy = True
if os.path.isdir(os.path.join(LOCAL_VDD, 'train', 'src')):
    n = len(os.listdir(os.path.join(LOCAL_VDD, 'train', 'src')))
    if n >= 280:
        print(f'[ok] VDD already at {LOCAL_VDD} ({n} train images)')
        needs_recopy = False
    else:
        print(f'[warn] VDD at {LOCAL_VDD} has only {n} images (corrupted), recopying.')
        shutil.rmtree(LOCAL_VDD)

if needs_recopy:
    if os.path.isdir(os.path.join(DRIVE_VDD, 'VDD', 'train')):
        SRC = os.path.join(DRIVE_VDD, 'VDD')
    elif os.path.isdir(os.path.join(DRIVE_VDD, 'train')):
        SRC = DRIVE_VDD
    else:
        raise RuntimeError(f'Drive layout not recognized at {DRIVE_VDD}')
    os.makedirs('/content/data', exist_ok=True)
    print(f'Copying VDD from {SRC} (~5-8 min, ~2 GB)')
    t0 = time.time()
    !cp -r "{SRC}" "{LOCAL_VDD}"
    print(f'  done in {time.time() - t0:.0f} s')

for split in ['train', 'val', 'test']:
    n = len(os.listdir(f'{LOCAL_VDD}/{split}/src'))
    print(f'  VDD/{split}: {n} images')

# ===== Step B: remove old foggy datasets (mask shape mismatch) =====
FOGGY_DATASETS = [
    ('VDD_foggy_medium_256', 'pix2pix_medium_baseline', 256),
    ('VDD_foggy_medium_768', 'pix2pix_medium_baseline', 768),
    ('VDD_foggy_dense_256',  'pix2pix_dense_baseline',  256),
    ('VDD_foggy_dense_768',  'pix2pix_dense_baseline',  768),
]
for ds_name, _, _ in FOGGY_DATASETS:
    p = f'/content/data/{ds_name}'
    if os.path.isdir(p):
        print(f'Removing old {p}')
        shutil.rmtree(p)

# ===== Step C: restore the two Pix2Pix generators from Drive =====
for run_name in ['pix2pix_medium_baseline', 'pix2pix_dense_baseline']:
    SRC = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{run_name}'
    DST = f'/content/fog-uav-robustness/outputs/runs/{run_name}'
    if os.path.isfile(f'{DST}/G_best.pth'):
        print(f'[ok] {run_name} generator already restored')
        continue
    if os.path.isdir(DST):
        shutil.rmtree(DST)
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copytree(SRC, DST)
    print(f'[ok] restored {run_name}')

# ===== Step D: install requirements (needed for the regen) =====
!pip install -q -r requirements-colab.txt

# ===== Step E: regenerate the 4 foggy datasets =====
for ds_name, gan_run, apply_size in FOGGY_DATASETS:
    print(f'\n=== Regenerating {ds_name} (apply_size={apply_size}) ===')
    !python src/inference/generate_foggy_vdd.py \
        --generator outputs/runs/{gan_run}/G_best.pth \
        --vdd-root {LOCAL_VDD} \
        --output-root /content/data/{ds_name} \
        --apply-size {apply_size} \
        --save-size 768 \
        --batch-size 4

# ===== Step F: verify mask shapes are now 768x768 =====
import cv2
print('\n=== Verifying mask shapes ===')
all_ok = True
for ds_name, _, _ in FOGGY_DATASETS:
    gt_dir = f'/content/data/{ds_name}/test/gt'
    if not os.path.isdir(gt_dir):
        print(f'  [missing] {gt_dir}')
        all_ok = False
        continue
    sample = sorted(os.listdir(gt_dir))[0]
    m = cv2.imread(f'{gt_dir}/{sample}', cv2.IMREAD_UNCHANGED)
    shape = m.shape if m is not None else None
    status = 'OK' if shape == (768, 768) else 'WRONG'
    print(f'  [{status}] {ds_name}: shape={shape}')
    if shape != (768, 768):
        all_ok = False
if not all_ok:
    raise RuntimeError('Some foggy masks have wrong shape, abort.')

# ===== Step G: backup the regenerated foggy datasets to Drive =====
DRIVE_DATA = '/content/drive/MyDrive/fog-uav-robustness/data'
for ds_name, _, _ in FOGGY_DATASETS:
    SRC = f'/content/data/{ds_name}'
    DST = f'{DRIVE_DATA}/{ds_name}'
    print(f'Backing up {ds_name} to Drive (overwriting old version)...')
    if os.path.isdir(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'  [ok]')

# ===== Final summary =====
print('\n=== Final file counts ===')
all_datasets = ['VDD'] + [d for d, _, _ in FOGGY_DATASETS]
for ds_name in all_datasets:
    LOCAL = f'/content/data/{ds_name}'
    if not os.path.isdir(LOCAL):
        print(f'  [missing] {ds_name}')
        continue
    counts = {split: len(os.listdir(f'{LOCAL}/{split}/src')) for split in ['train','val','test']}
    print(f'  {ds_name}: train={counts["train"]}, val={counts["val"]}, test={counts["test"]}')

## 5. Restore the U-Net v2 checkpoint from Drive

We need `outputs/runs/unet_resnet34_clean_v2_weighted/best.pth` (Phase 1).

In [ ]:
import os, shutil

RUN = 'unet_resnet34_clean_v2_weighted'
SRC = f'{DRIVE}/outputs/{RUN}'
DST = f'/content/fog-uav-robustness/outputs/runs/{RUN}'

if os.path.isfile(f'{DST}/best.pth'):
    print(f'[skip] U-Net v2 already at {DST}')
elif os.path.isdir(SRC):
    if os.path.isdir(DST):
        shutil.rmtree(DST)
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copytree(SRC, DST)
    print(f'[ok] restored U-Net v2 from {SRC}')
    !ls -la "{DST}" | head -8
else:
    raise RuntimeError(f'U-Net v2 not on Drive! Expected at {SRC}.')

## 6. Install Python dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

## 7. Launch TensorBoard

Open TB now. Each evaluation will append `<dataset>/test/*` scalars to the
U-Net's tb folder, so we'll see all 5 evaluations side-by-side.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/runs

## 8. Sanity check on VDD clean (test split)

First reproduce the Phase 1 number (test mIoU = 0.7168) to confirm everything
is consistent before running the foggy evaluations.

In [ ]:
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_clean_v2_weighted/best.pth \
    --data-root /content/data/VDD \
    --split test \
    --image-size 768 \
    --batch-size 4 \
    --output outputs/runs/unet_resnet34_clean_v2_weighted/test_results_clean.json

Expected output: `mIoU ~ 0.7168` (within rounding).

## 9. Robustness evaluation: U-Net on the 4 foggy variants

We run `evaluate.py` once per foggy variant and save the results into named
JSON files. Each run takes ~15-30 seconds on T4.

In [ ]:
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_clean_v2_weighted/best.pth \
    --data-root /content/data/VDD_foggy_medium_256 \
    --split test \
    --image-size 768 \
    --batch-size 4 \
    --output outputs/runs/unet_resnet34_clean_v2_weighted/test_results_foggy_medium_256.json \
    --no-tb

In [ ]:
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_clean_v2_weighted/best.pth \
    --data-root /content/data/VDD_foggy_medium_768 \
    --split test \
    --image-size 768 \
    --batch-size 4 \
    --output outputs/runs/unet_resnet34_clean_v2_weighted/test_results_foggy_medium_768.json \
    --no-tb

In [ ]:
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_clean_v2_weighted/best.pth \
    --data-root /content/data/VDD_foggy_dense_256 \
    --split test \
    --image-size 768 \
    --batch-size 4 \
    --output outputs/runs/unet_resnet34_clean_v2_weighted/test_results_foggy_dense_256.json \
    --no-tb

In [ ]:
!python src/evaluation/evaluate.py \
    --checkpoint outputs/runs/unet_resnet34_clean_v2_weighted/best.pth \
    --data-root /content/data/VDD_foggy_dense_768 \
    --split test \
    --image-size 768 \
    --batch-size 4 \
    --output outputs/runs/unet_resnet34_clean_v2_weighted/test_results_foggy_dense_768.json \
    --no-tb

## 10. Aggregate results into one comparison table

Read the 5 JSON files and produce the table that goes into the report.

In [ ]:
import json, os

BASE = '/content/fog-uav-robustness/outputs/runs/unet_resnet34_clean_v2_weighted'
files = [
    ('clean',         f'{BASE}/test_results_clean.json'),
    ('medium @256',   f'{BASE}/test_results_foggy_medium_256.json'),
    ('medium @768',   f'{BASE}/test_results_foggy_medium_768.json'),
    ('dense @256',    f'{BASE}/test_results_foggy_dense_256.json'),
    ('dense @768',    f'{BASE}/test_results_foggy_dense_768.json'),
]

results = {}
for label, path in files:
    if not os.path.isfile(path):
        print(f'[missing] {label}: {path}')
        continue
    with open(path) as f:
        results[label] = json.load(f)

# Aggregate metric table
if 'clean' not in results:
    raise RuntimeError('No clean results found, abort.')
clean_miou = results['clean']['mIoU']

print('=' * 78)
print('PHASE 4: U-Net robustness under GAN-generated fog (test set)')
print('=' * 78)
print(f'{"variant":18s}  {"mIoU":>8s}  {"F1":>8s}  {"acc":>8s}  {"loss":>8s}  {"dmIoU":>10s}')
print('-' * 78)
for label, _ in files:
    if label not in results:
        continue
    r = results[label]
    drop = r['mIoU'] - clean_miou
    drop_pct = 100 * drop / clean_miou
    drop_str = f'{drop:+.4f}' if label == 'clean' else f'{drop:+.4f} ({drop_pct:+.1f}%)'
    print(f'{label:18s}  {r["mIoU"]:>8.4f}  {r["F1"]:>8.4f}  {r["accuracy"]:>8.4f}  {r["loss"]:>8.4f}  {drop_str}')

print()
print('Per-class IoU comparison:')
print('-' * 78)
header = f'{"class":14s}  ' + '  '.join(f'{lbl:>11s}' for lbl, _ in files if lbl in results)
print(header)
for cls in results['clean']['per_class_iou']:
    row = f'{cls:14s}  ' + '  '.join(
        f'{results[lbl]["per_class_iou"][cls]:>11.4f}'
        for lbl, _ in files if lbl in results
    )
    print(row)

In [ ]:
# Pretty per-class drop visualization (bars relative to clean)
import matplotlib.pyplot as plt
import numpy as np

if 'clean' in results:
    classes = list(results['clean']['per_class_iou'].keys())
    variants = [lbl for lbl, _ in files if lbl in results and lbl != 'clean']

    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(classes))
    width = 0.18

    # Clean baseline (always shown)
    clean_vals = [results['clean']['per_class_iou'][c] for c in classes]
    ax.bar(x - 2*width, clean_vals, width, label='clean', color='black', alpha=0.7)

    colors = ['#5DADE2', '#3498DB', '#E67E22', '#C0392B']
    for i, v in enumerate(variants):
        vals = [results[v]['per_class_iou'][c] for c in classes]
        ax.bar(x + (i-1)*width, vals, width, label=v, color=colors[i % 4])

    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=20)
    ax.set_ylabel('IoU')
    ax.set_title('Per-class IoU: clean vs foggy variants (test set)')
    ax.legend(loc='upper left', ncols=3, fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    out = '/content/fog-uav-robustness/outputs/figures/phase4_per_class_iou.png'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    plt.savefig(out, dpi=80, bbox_inches='tight')
    plt.show()
    print(f'\nFigure saved: {out}')

## 11. Backup all results to Drive

Save the JSONs and the per-class figure to Drive.

In [ ]:
import shutil, os

RUN = 'unet_resnet34_clean_v2_weighted'
SRC = f'/content/fog-uav-robustness/outputs/runs/{RUN}'
DST = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{RUN}'

# Copy any test_results_*.json that doesn't yet exist on Drive (or update them)
for fname in os.listdir(SRC):
    if fname.startswith('test_results') and fname.endswith('.json'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(DST, fname))
        print(f'[ok] {fname}')

# Also re-back up the tb/ folder so the merged scalars are saved
TB_SRC = f'{SRC}/tb'
TB_DST = f'{DST}/tb'
if os.path.isdir(TB_DST):
    shutil.rmtree(TB_DST)
shutil.copytree(TB_SRC, TB_DST)
print(f'[ok] tb/ backed up')

# Copy figure
FIG_SRC = '/content/fog-uav-robustness/outputs/figures/phase4_per_class_iou.png'
FIG_DST = '/content/drive/MyDrive/fog-uav-robustness/outputs/figures/phase4_per_class_iou.png'
if os.path.isfile(FIG_SRC):
    os.makedirs(os.path.dirname(FIG_DST), exist_ok=True)
    shutil.copy(FIG_SRC, FIG_DST)
    print(f'[ok] figure backed up')

print('\nAll results on Drive:')
!ls -la "{DST}" | grep test_results

## What's next (Phase 5)

If the U-Net's mIoU dropped significantly on the foggy datasets (which is what
we expect), then in Phase 5 we will:

1. Build a 'mixed' training set that includes both clean VDD and foggy VDD
   (or replace clean with one of the foggy variants)
2. Re-train the U-Net on this augmented set
3. Test the re-trained model on **all** variants (clean + 4 foggy)
4. Compare: did the re-training recover the lost mIoU on foggy without hurting
   clean too much?

This will be the answer to the project's central question: 'is GAN-augmented
training a viable strategy to make UAV semantic segmentation robust to fog?'